In [9]:
import pandas as pd

# Load the 'train.csv' file into a pandas DataFrame
train_df = pd.read_csv('/content/drive/MyDrive/diplomado_cm_ia/model_ensemble/train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/diplomado_cm_ia/model_ensemble/test.csv')



In [12]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import KFold
from sklearn.base import clone
import pandas as pd

In [11]:
def wrapper_feature_selection(X,y,model,search_method='forward',k_features=20,cv=10,scoring='r2',n_jobs=-1):
  kf = KFold(n_splits=cv, shuffle=True, random_state=42)
  estimator = clone(model)

  selector = SequentialFeatureSelector(
            estimator=estimator,
            n_features_to_select=k_features,
            direction=search_method,
            scoring=scoring,
            cv=kf,
            n_jobs=n_jobs
        )
  selector.fit(X, y)
  selected_features = X.columns[selector.get_support()]

  return {"selected_features": selected_features,
          "X_selected": X[selected_features],
          "selector": selector
          }

In [ ]:
X = train_df.drop('Class', axis=1)
y = train_df['Class']

met = 'forward'
score = 'accuracy'

modelos = {
    # 'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='linear', random_state=42), # Add random_state for reproducibility
    # 'knn': KNeighborsClassifier(n_neighbors=5),
}

results = {}
output_dir = '/content/drive/MyDrive/diplomado_cm_ia/model_ensemble/'

for mod_name, model in modelos.items():
  print(f"Performing feature selection for {mod_name}...")
  res = wrapper_feature_selection(
                    X=X, y=y, model=model,
                    search_method=met, k_features=5,
                    cv=10, scoring=score
                )
  results[mod_name] = res
  print(f"Selected features for {mod_name}: {res['selected_features'].tolist()}\n")

  # Save selected train data
  selected_features_names = res['selected_features']
  train_selected_df_model = pd.concat([X[selected_features_names], y], axis=1)
  train_output_path = output_dir + f'train_selected_{mod_name}.csv'
  train_selected_df_model.to_csv(train_output_path, index=False)
  print(f"Selected training data for {mod_name} saved to: {train_output_path}")

  # Save selected test data
  if 'Class' in test_df.columns:
      X_test_original = test_df.drop('Class', axis=1)
      y_test_original = test_df['Class']
  else:
      X_test_original = test_df
      y_test_original = None

  features_present_in_test = [f for f in selected_features_names if f in X_test_original.columns]
  X_test_selected = X_test_original[features_present_in_test]

  if y_test_original is not None:
      test_selected_df_model = pd.concat([X_test_selected, y_test_original], axis=1)
  else:
      test_selected_df_model = X_test_selected

  test_output_path = output_dir + f'test_selected_{mod_name}.csv'
  test_selected_df_model.to_csv(test_output_path, index=False)
  print(f"Selected test data for {mod_name} saved to: {test_output_path}\n")

# You can access the results like this:
# print(results['RandomForest']['selected_features'])
# print(results['SVM']['X_selected'].head())

Performing feature selection for SVM...
